<a href="https://colab.research.google.com/github/marwa189/cervical-cancer-risk-prediction/blob/main/Cervical_Cancer_Prediction_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cervical Cancer Biopsy Prediction Using Machine Learning

## Objective

Develop and evaluate machine learning models for predicting cervical cancer biopsy outcomes using demographic, behavioral, and clinical screening data.

## Methodology

1. Exploratory Data Analysis (EDA)
2. Data cleaning and preprocessing
3. Missing value imputation
4. Train-test splitting
5. Class imbalance handling using SMOTE
6. Model training and evaluation

   * Logistic Regression
   * Random Forest
   * Gradient Boosting
7. Hyperparameter tuning
8. PCA evaluation
9. Feature importance analysis
10. Cross-validation
11. Model comparison and selection

## Key Findings

* Gradient Boosting with SMOTE achieved the best overall performance.
* Hyperparameter tuning improved model robustness but not test performance.
* PCA reduced predictive performance.
* The Schiller test was the most influential feature.
* EDA revealed missing values and class imbalance, guiding preprocessing decisions.

## Final Model

Gradient Boosting with SMOTE was selected as the final model due to its strong predictive performance and good generalization.


In [ ]:
## preprocessing:

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# read dataset
df=pd.read_csv("risk_factors_cervical_cancer.csv")


In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
(df == "?").sum()

In [ ]:
# hidden missing values
df=df.replace("?",np.nan)

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
# change to numeric instead of obj caused by ?
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df.info()

In [ ]:
missing_percentage = (df.isna().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)

In [ ]:
df["Biopsy"].value_counts()

In [ ]:
sns.countplot(x="Biopsy",data=df)
plt.title("Biopsy distribution")
plt.show()

# Exploratory Data Analysis (EDA)
Generate an automated profiling report to explore data distributions, missing values, correlations, and feature characteristics.

In [ ]:
! pip install ydata-profiling

In [ ]:
# Generate YData Profiling Report
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Cervical Cancer Dataset Profiling Report",
    explorative=True
)

profile.to_notebook_iframe()

In [ ]:
# Drop columns with extremely high missing values
df.drop(columns=[
    "STDs: Time since first diagnosis",
    "STDs: Time since last diagnosis"
], inplace=True)

In [ ]:
# zero variance
constant_cols = [
    "STDs:cervical condylomatosis",
    "STDs:AIDS"
]

df.drop(columns=constant_cols, inplace=True)

In [ ]:
#duplicate rows
df.duplicated().sum()

In [ ]:
#delete duplicate rows
df.drop_duplicates(inplace=True)

In [ ]:
missing_percentage = (df.isna().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)

In [ ]:
#measure skewness
numerical_cols = [
    "Age",
    "Number of sexual partners",
    "First sexual intercourse",
    "Num of pregnancies",
    "Smokes (years)",
    "Smokes (packs/year)",
    "IUD (years)",
    "Hormonal Contraceptives (years)",
    "STDs (number)"
]

df[numerical_cols].skew()

Data Splitting

In [ ]:
# define features and target
X = df.drop("Biopsy", axis=1)
y = df["Biopsy"]

In [ ]:
#train test split
# stratify=y: Because dataset is imbalanced
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

Distinguish Numeric vs Categoric

In [ ]:
# numerical columns
numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns.tolist()

# categorical columns (0/1 features)
categorical_cols = []

for col in numeric_cols:
    unique_vals = set(df[col].dropna().unique())

    if unique_vals.issubset({0, 1}):
        categorical_cols.append(col)

# remove target if present
if "Biopsy" in categorical_cols:
    categorical_cols.remove("Biopsy")

# numerical features only
numeric_cols = [
    col for col in numeric_cols
    if col not in categorical_cols and col != "Biopsy"
]

print("Numerical:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

In [ ]:
print("Numerical:")
print(numeric_cols)

print("\nCategorical:")
print(categorical_cols)

In [ ]:
# seperate train subsets
X_train_num = X_train[numeric_cols]
X_train_cat = X_train[categorical_cols]
X_test_num = X_test[numeric_cols].copy()
X_test_cat = X_test[categorical_cols].copy()



imputation

In [ ]:
# Impute numerical columns with KNN
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=5)

X_train_num = pd.DataFrame(
    knn_imputer.fit_transform(X_train_num),
    columns=numeric_cols,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    knn_imputer.transform(X_test_num),
    columns=numeric_cols,
    index=X_test.index
)

In [ ]:
# Impute categorical columns with Most Frequent
from sklearn.impute import SimpleImputer

cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train_cat),
    columns=categorical_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test_cat),
    columns=categorical_cols,
    index=X_test.index
)

In [ ]:
# merge them back
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

In [ ]:
# restore original column order
X_train = X_train[numeric_cols + categorical_cols]
X_test = X_test[numeric_cols + categorical_cols]

In [ ]:
X_train.isna().sum().sum()

In [ ]:
X_train[categorical_cols].head()

In [ ]:
X_train[categorical_cols].tail()

feature selection

In [ ]:
# matual info
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train, y_train)

In [ ]:
mi_scores = pd.Series(mi_scores, index=X_train.columns)
mi_scores.sort_values(ascending=False)

In [ ]:
# drop columns with zero mi score ?
zero_mi_cols = mi_scores[mi_scores == 0].index.tolist()
print(zero_mi_cols)

In [ ]:
for col in zero_mi_cols:
    print("\n", col)
    print(df[col].value_counts())

Several medically relevant variables obtained very low or zero Mutual Information scores. Further investigation showed that many of these variables were extremely sparse (e.g., STDs:HPV appeared in only 2 records). Therefore, despite their known clinical importance, the dataset did not contain enough observations for these features to contribute significantly to statistical feature selection methods.

In [ ]:
pd.crosstab(df["Smokes"], df["Biopsy"])

In [ ]:
pd.crosstab(df["Num of pregnancies"], df["Biopsy"])

Although several features obtained near-zero Mutual Information scores, further analysis showed that some still exhibited meaningful differences in biopsy positivity rates, indicating that MI alone is insufficient for feature selection, especially in sparse and imbalanced medical datasets.

# Baseline Models

After completing preprocessing and exploratory data analysis, baseline machine learning models will be trained to establish initial performance benchmarks.

The project begins with:
- Logistic Regression
- Random Forest
- Gradient Boosting

These models provide a comparison between interpretable linear modeling and robust tree-based learning approaches.

In [ ]:
##Logistic Regression

In [ ]:
# Scale features for Logistic Regression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# train logistic regression
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

lr_model.fit(X_train_scaled, y_train)

In [ ]:
#import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [ ]:
# predict on train values
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_prob_lr_train = lr_model.predict_proba(X_train_scaled)[:, 1]

In [ ]:
# print matrix for train prediction
print("Accuracy:", accuracy_score(y_train, y_pred_lr_train))
print("Precision:", precision_score(y_train, y_pred_lr_train))
print("Recall:", recall_score(y_train, y_pred_lr_train))
print("F1-score:", f1_score(y_train, y_pred_lr_train))
print("ROC-AUC:", roc_auc_score(y_train, y_prob_lr_train))

In [ ]:
# predict on test values
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# print matrices on test prediction
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1-score:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

In [ ]:
# confusion matrix
confusion_matrix(y_test, y_pred_lr)

The F1-score 0.75 reflects the balance between recall and precision, providing a more reliable evaluation metric for this imbalanced healthcare dataset than accuracy alone.

In [ ]:
## Random Forest

In [ ]:
# import random forest classifier
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

In [ ]:
#predict on train
y_pred_rf = rf_model.predict(X_train)
y_prob_rf = rf_model.predict_proba(X_train)[:, 1]

In [ ]:
# print matrices
print("Accuracy:", accuracy_score(y_train, y_pred_rf))
print("Precision:", precision_score(y_train, y_pred_rf))
print("Recall:", recall_score(y_train, y_pred_rf))
print("F1-score:", f1_score(y_train, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_train, y_prob_rf))

In [ ]:
# train on test values
y_pred_rf_train = rf_model.predict(X_test)
y_prob_rf_train = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
# predict on test
y_pred_rf_test = rf_model.predict(X_test)
y_prob_rf_test = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
# print matrices
print("Accuracy:", accuracy_score(y_test, y_pred_rf_test))
print("Precision:", precision_score(y_test, y_pred_rf_test))
print("Recall:", recall_score(y_test, y_pred_rf_test))
print("F1-score:", f1_score(y_test, y_pred_rf_test))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf_test))

The Random Forest is memorizing data on train and unable to generalize on new test data

## Gradient Boosting Baseline Model

Gradient Boosting is added as a more advanced ensemble model. Unlike Random Forest, which builds trees independently, Gradient Boosting builds trees sequentially, where each new tree tries to correct the errors of the previous ones.

This model is evaluated first without SMOTE or hyperparameter tuning to understand its baseline behavior on the preprocessed dataset.

In [ ]:
# import gradient boosting
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
# create the model
gb_model = GradientBoostingClassifier(
    random_state=42
)

# train the model
gb_model.fit(X_train, y_train)

In [ ]:
# predict on train
y_pred_gb_train = gb_model.predict(X_train)
y_prob_gb_train = gb_model.predict_proba(X_train)[:, 1]

In [ ]:
# print matrices
print("Accuracy:", accuracy_score(y_train, y_pred_gb_train))
print("Precision:", precision_score(y_train, y_pred_gb_train))
print("Recall:", recall_score(y_train, y_pred_gb_train))
print("F1-score:", f1_score(y_train, y_pred_gb_train))
print("ROC-AUC:", roc_auc_score(y_train, y_prob_gb_train))

In [ ]:
# train on test
y_pred_gb= gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

In [ ]:
# predict
y_pred_gb = gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

In [ ]:
# print evaluation matrices
print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Precision:", precision_score(y_test, y_pred_gb))
print("Recall:", recall_score(y_test, y_pred_gb))
print("F1-score:", f1_score(y_test, y_pred_gb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_gb))
#

In [ ]:
# confusion matrix
confusion_matrix(y_test, y_pred_gb)

Model is overfitting and unable to generalize

SMOTE was added as an extra experiment because the dataset is highly imbalanced and positive biopsy cases are clinically important. Its effect was evaluated by comparing model performance before and after balancing, especially recall, false negatives, precision, and F1-score.

In [ ]:
# import smote
from imblearn.over_sampling import SMOTE

In [ ]:
# create smote object
smote = SMOTE(random_state=42)

In [ ]:
# apply smote only on training data
# for logistic regression
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
# verify new balance
y_train_smote.value_counts()
#

In [ ]:
#visualize class balance
sns.countplot(x=y_train_smote)
plt.title("Class Balance After SMOTE")
plt.show()
#

In [ ]:
plt.figure(figsize=(8,5))

sns.kdeplot(X_train_scaled[:,0], label="Original Scaled", fill=True)
sns.kdeplot(X_train_smote[:,0], label="SMOTE", fill=True)

plt.legend()
plt.title("Feature Distribution Before and After SMOTE")
plt.show()

In [ ]:
#train with X_train_smote and y_train_smote
lr_smote = LogisticRegression(
    random_state=42,
    max_iter=1000
)
lr_smote.fit(X_train_smote, y_train_smote)

In [ ]:
#predict on train
y_pred_lr_smote_train = lr_smote.predict(X_train_scaled)
y_prob_lr_smote_train = lr_smote.predict_proba(X_train_scaled)[:, 1]

In [ ]:
#print matrices
print("Accuracy:", accuracy_score(y_train, y_pred_lr_smote_train))
print("Precision:", precision_score(y_train, y_pred_lr_smote_train))
print("Recall:", recall_score(y_train, y_pred_lr_smote_train))
print("F1-score:", f1_score(y_train, y_pred_lr_smote_train))

In [ ]:
# pridect
# Use ORIGINAL untouched test set
y_pred_lr_smote = lr_smote.predict(X_test_scaled)
y_prob_lr_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]
#

In [ ]:
# print evaluayion matrices
print("Accuracy:", accuracy_score(y_test, y_pred_lr_smote))
print("Precision:", precision_score(y_test, y_pred_lr_smote))
print("Recall:", recall_score(y_test, y_pred_lr_smote))
print("F1-score:", f1_score(y_test, y_pred_lr_smote))

Model is generalizing well and 0.78 is acceptable and higher than before

In [ ]:
# check confusion matrix before and after smote
print("Without SMOTE")
print(confusion_matrix(y_test, y_pred_lr))

print("\nWith SMOTE")
print(confusion_matrix(y_test, y_pred_lr_smote))

For Gradient Boost:

In [ ]:
# crete smote to inscaled train data for gradient boost

smote_gr = SMOTE(random_state=42)

X_train_smote_gb, y_train_smote_gb = smote_gr.fit_resample(X_train, y_train)

In [ ]:
# visualize class balance
sns.countplot(x=y_train_smote_gb)
plt.title("Class Balance After SMOTE")
plt.show()

In [ ]:
# create a gradient booster model with SMOTE
gb_smote = GradientBoostingClassifier(random_state=42)

gb_smote.fit(X_train_smote_gb, y_train_smote_gb)

In [ ]:
y_pred_gb_smote_train= gb_smote.predict(X_train)
y_prob_gb_smote_train = gb_smote.predict_proba(X_train)[:, 1]

In [ ]:
# print matrices
print("Accuracy:", accuracy_score(y_train, y_pred_gb_smote_train))
print("Precision:", precision_score(y_train, y_pred_gb_smote_train))
print("Recall:", recall_score(y_train, y_pred_gb_smote_train))
print("F1-score:", f1_score(y_train, y_pred_gb_smote_train))


In [ ]:
# confusion matrix
confusion_matrix(y_train, y_pred_gb_smote_train)

In [ ]:
# predict on test
y_pred_gb_smote = gb_smote.predict(X_test)
y_prob_gb_smote = gb_smote.predict_proba(X_test)[:, 1]

In [ ]:
# print matrices
print("Accuracy:", accuracy_score(y_test, y_pred_gb_smote))
print("Precision:", precision_score(y_test, y_pred_gb_smote))
print("Recall:", recall_score(y_test, y_pred_gb_smote))
print("F1-score:", f1_score(y_test, y_pred_gb_smote))

Strong predictive performance while maintaining an acceptable train-test gap of about 9 percentage points.




Training and testing F1-scores are compared to investigate potential overfitting after applying SMOTE and Gradient Boosting.

In [ ]:
y_train_pred_lr = lr_model.predict(X_train_scaled)
train_f1_lr = f1_score(y_train, y_train_pred_lr)

y_train_pred_lr_smote = lr_smote.predict(X_train_smote)
train_f1_lr_smote = f1_score(y_train_smote, y_train_pred_lr_smote)

y_train_pred_gb = gb_model.predict(X_train)
train_f1_gb = f1_score(y_train, y_train_pred_gb)

In [ ]:
# comparison table F1 train and test before and after smote for logistic regression and gradient boosting

import pandas as pd
from sklearn.metrics import f1_score

# --- Calculations for Logistic Regression without SMOTE ---
y_train_pred_lr = lr_model.predict(X_train_scaled)
train_f1_lr = f1_score(y_train, y_train_pred_lr)
test_f1_lr = f1_score(y_test, y_pred_lr)

# --- Calculations for Logistic Regression with SMOTE ---
# For train F1, predict on original scaled training data and compare with original y_train
y_pred_lr_smote_train_orig_y = lr_smote.predict(X_train_scaled)
train_f1_lr_smote_orig_y = f1_score(y_train, y_pred_lr_smote_train_orig_y)
test_f1_lr_smote = f1_score(y_test, y_pred_lr_smote)

# --- Calculations for Gradient Boosting without SMOTE ---
y_train_pred_gb = gb_model.predict(X_train)
train_f1_gb = f1_score(y_train, y_train_pred_gb)
test_f1_gb = f1_score(y_test, y_pred_gb)

# --- Calculations for Gradient Boosting with SMOTE ---
# For train F1, predict on original training data and compare with original y_train
y_pred_gb_smote_train_orig_y = gb_smote.predict(X_train)
train_f1_gb_smote_orig_y = f1_score(y_train, y_pred_gb_smote_train_orig_y)
test_f1_gb_smote = f1_score(y_test, y_pred_gb_smote)

comparison_table = pd.DataFrame({
    "Model": [
        "Logistic Regression (No SMOTE)",
        "Logistic Regression (with SMOTE)",
        "Gradient Boosting (No SMOTE)",
        "Gradient Boosting (with SMOTE)"
    ],
    "Train F1": [
        train_f1_lr,
        train_f1_lr_smote_orig_y,
        train_f1_gb,
        train_f1_gb_smote_orig_y
    ],
    "Test F1": [
        test_f1_lr,
        test_f1_lr_smote,
        test_f1_gb,
        test_f1_gb_smote
    ]
})

# Calculate the 'Gap (pp)' column
comparison_table['Gap (pp)'] = (comparison_table['Train F1'] - comparison_table['Test F1']) * 100

print(comparison_table.to_markdown(index=False))

Gradient Boosting with SMOTE was selected as the final model due to its superior test F1-score and its favorable balance between performance and generalization.

To assess the robustness of the model and rule out train-test split bias, 5-fold cross-validation was performed.

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
])

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("Fold scores:", scores)
print("Mean F1:", scores.mean())
print("Std:", scores.std())

The model achieved F1-scores ranging from 0.59 to 0.80 across the five folds, with a mean F1-score of 0.67 and a standard deviation of 0.075. This indicates reasonably consistent performance across different data partitions, although the higher test-set F1-score (0.87) suggests that the selected test split was more favorable than average.

## Hyperparameter Tuning

Gradient Boosting with SMOTE was selected as the best-performing model. Hyperparameter tuning is performed using GridSearchCV to optimize the F1-score, providing a balance between recall and precision while reducing the risk of false negatives.

In [ ]:
# imports
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier



In [ ]:
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("gb", GradientBoostingClassifier(random_state=42))
])

In [ ]:
# parameter grid
param_grid =  {
    "gb__n_estimators": [50, 100, 200],
    "gb__learning_rate": [0.01, 0.05, 0.1],
    "gb__max_depth": [2, 3, 4]

}

In [ ]:
grid_search=GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
# best parameters
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

In [ ]:
# best score
best_score = grid_search.best_score_
print("Best Score (F1):", best_score)

In [ ]:
# best model
best_model = grid_search.best_estimator_
print("Best Model:", best_model)

In [ ]:
y_pred_best = grid_search.predict(X_test)

print("F1:", f1_score(y_test, y_pred_best))

Hyperparameter tuning produced a more robust model, even though its test F1-score was slightly lower than the baseline result.

PCA experiment

In [ ]:
# PCA is sensitive to feature scales
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# keep 95% of variance
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
# Check dimensionality reduction
print("Original features:", X_train.shape[1])
print("PCA features:", X_train_pca.shape[1])

In [ ]:
# apply smote on pca
smote_pca = SMOTE(random_state=42)

X_train_pca_smote, y_train_pca_smote = smote_pca.fit_resample(X_train_pca, y_train)
#

In [ ]:
# train gradient boosting model with pca and smote
gb_pca_smote = GradientBoostingClassifier(random_state=42)

gb_pca_smote.fit(X_train_pca_smote, y_train_pca_smote)

In [ ]:
# evaluate
y_pred_pca_smote = gb_pca_smote.predict(X_test_pca)

print("F1:", f1_score(y_test, y_pred_pca_smote))

In [ ]:
# evaluate on train
y_pred_pca_smote_train = gb_pca_smote.predict(X_train_pca)

print("F1:", f1_score(y_train, y_pred_pca_smote_train))

In [ ]:
# compare with and without pca
print("Without PCA:")
print(confusion_matrix(y_test, y_pred_gb_smote))

print("\nWith PCA:")
print(confusion_matrix(y_test, y_pred_pca_smote))
#

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")

Approximately 20 principal components were required to retain 95% of the variance, and PCA reduced model performance, suggesting that the original feature space contained valuable predictive information.

Feature Importance

In [ ]:
# Evaluate the contribution of each feature to the final Gradient Boosting + SMOTE model

importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb_smote.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

importance_df.head(10)

In [ ]:
# Plot top 10 feature importances

top10 = importance_df.head(10)

plt.figure(figsize=(8,5))
plt.barh(top10["Feature"], top10["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 10 Most Important Features")
plt.tight_layout()
plt.show()

Model Comparison Bar Chart

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(comparison_df["Model"], comparison_df["Test F1"])
plt.ylabel("Test F1 Score")
plt.xlabel("Model")
plt.title("Model Performance Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Final Conclusion**

Gradient Boosting with SMOTE achieved the best overall performance (Test F1 = 0.87) with a train-test gap of approximately 9.6 percentage points, indicating good generalization. Feature importance analysis revealed that diagnostic screening tests, particularly the Schiller test, contributed most strongly to prediction. PCA reduced performance, suggesting that the original features contained valuable discriminative information. Although hyperparameter tuning produced a slightly lower test F1-score (0.83), cross-validation indicated that the tuned model provided a more robust estimate of real-world performance.